## (I) Understand the task and inspect the data
### 1.1 Task and field description
Load the dataset and understand each field. `id` and `breath_id` are only used to organize data, not as prediction features. Train/validation/test sets are all split internally from `train.csv`; the competition `test.csv` is not used for metric computation.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path


SEED = 42
import os
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = Path(os.environ.get("VENTILATOR_DATA_PATH", ROOT_DIR / "data" / "train.csv"))
WEIGHT_DIR = ROOT_DIR / "weights"
FIG_DIR = ROOT_DIR / "figures"
WEIGHT_DIR.mkdir(exist_ok=True); FIG_DIR.mkdir(exist_ok=True)
np.random.seed(SEED)
from matplotlib import font_manager
_fonts = {f.name for f in font_manager.fontManager.ttflist}
_cjk = next((f for f in ["SimHei","Microsoft YaHei","Noto Sans CJK SC"] if f in _fonts), "DejaVu Sans")
plt.rcParams["font.sans-serif"] = [_cjk]
plt.rcParams['axes.unicode_minus'] = False

# Load data
if not Path(DATA_PATH).is_file():
    raise FileNotFoundError(f"Data file not found: {Path(DATA_PATH).resolve()}. Please set DATA_PATH to the actual train.csv path.")
df = pd.read_csv(DATA_PATH)
print("Raw data shape:", df.shape)
print(df.columns.tolist())
print(df.head())







### 1.2 Data validation and visualization
Sort by `breath_id` and `time_step`, check missing values and duplicate records, verify each breathing cycle has 80 time steps, then plot the u_in, u_out and pressure time series for several cycles.


In [ ]:
# 1. Missing value check
print("===== Missing value counts =====")
print(df.isnull().sum())

# 2. Duplicate record check
print("\n===== Number of duplicate rows =====")
print("Duplicate rows:", df.duplicated().sum())

# 3. Sort by breath_id and time_step to keep correct ordering
df = df.sort_values(by=["breath_id", "time_step"]).reset_index(drop=True)

# 4. Count time steps per cycle (expected: 80)
breath_len = df.groupby("breath_id").size()
print("\n===== Time-step count per breath_id =====")
print("Cycle length distribution:", breath_len.value_counts().sort_index())
print("Min length:", breath_len.min(), "Max length:", breath_len.max())
print("All cycles are 80 steps:", (breath_len == 80).all())

# 5. Inspect the distribution of R/C parameter combinations
print("\n===== R/C parameter combinations =====")
print(df[["R","C"]].drop_duplicates())




In [ ]:
# Plot u_in, u_out and pressure time series for several breathing cycles
sample_breath_ids = breath_len.index[:4]
plt.figure(figsize=(14, 8))

for i, bid in enumerate(sample_breath_ids):
    sub_df = df[df["breath_id"] == bid]
    plt.subplot(2, 2, i+1)
    plt.plot(sub_df["time_step"], sub_df["u_in"], label="u_in (inhalation valve)", c="#2E86AB")
    plt.plot(sub_df["time_step"], sub_df["u_out"], label="u_out (exhalation valve)", c="#A23B72")
    plt.plot(sub_df["time_step"], sub_df["pressure"], label="pressure", c="#F18F01")
    plt.title(f"breath_id = {bid}, R={sub_df['R'].iloc[0]}, C={sub_df['C'].iloc[0]}")
    plt.xlabel("time_step")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()



## (II) Grouped split and feature engineering
### 2.1 Dataset split
Sample a fixed number of complete `breath_id`s with R/C stratification, split 70% train / 15% validation / 15% local test, ensure the three sets have disjoint `breath_id`s, and save the split indices.


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import torch
from torch.utils.data import Dataset, DataLoader
np.random.seed(SEED)
torch.manual_seed(SEED)

sample_num = 8000
breath_rc = df[["breath_id","R","C"]].drop_duplicates().reset_index(drop=True)
sample_num = min(sample_num, len(breath_rc))
breath_sample = breath_rc.sample(n=sample_num, random_state=SEED).reset_index(drop=True)
selected_breath_ids = breath_sample["breath_id"].values
df_sub = df[df["breath_id"].isin(selected_breath_ids)].copy()
print(f"Sampling done, number of breathing cycles selected: {len(selected_breath_ids)}")
print("R/C combinations after sampling:\n", df_sub[["R","C"]].drop_duplicates())

all_bids = df_sub["breath_id"].unique()
strata_map = dict(zip(breath_sample["breath_id"],
                      breath_sample["R"].astype(str) + "_" + breath_sample["C"].astype(str)))
bid_train, bid_temp = train_test_split(all_bids, test_size=0.3, random_state=SEED, shuffle=True,
                                       stratify=[strata_map[b] for b in all_bids])
bid_val, bid_test = train_test_split(bid_temp, test_size=0.5, random_state=SEED, shuffle=True,
                                     stratify=[strata_map[b] for b in bid_temp])
print(f"Train breaths: {len(bid_train)}, val breaths: {len(bid_val)}, local test breaths: {len(bid_test)}")
df_train = df_sub[df_sub["breath_id"].isin(bid_train)].copy()
df_val = df_sub[df_sub["breath_id"].isin(bid_val)].copy()
df_test = df_sub[df_sub["breath_id"].isin(bid_test)].copy()

cont_feats = ["time_step", "u_in"]
cat_feats = ["R", "C"]
target_col = "pressure"

scaler = StandardScaler()
df_train[cont_feats] = scaler.fit_transform(df_train[cont_feats])
df_val[cont_feats] = scaler.transform(df_val[cont_feats])
df_test[cont_feats] = scaler.transform(df_test[cont_feats])

ohe = OneHotEncoder(sparse_output=False, drop=None)
rc_train = ohe.fit_transform(df_train[cat_feats])
rc_val = ohe.transform(df_val[cat_feats])
rc_test = ohe.transform(df_test[cat_feats])

X_train = np.concatenate([df_train[cont_feats].values, df_train[["u_out"]].values, rc_train], axis=1)
X_val = np.concatenate([df_val[cont_feats].values, df_val[["u_out"]].values, rc_val], axis=1)
X_test = np.concatenate([df_test[cont_feats].values, df_test[["u_out"]].values, rc_test], axis=1)

new_cols = cont_feats + ["u_out"] + [f"rc_{i}" for i in range(rc_train.shape[1])]
df_train[new_cols] = X_train
df_val[new_cols] = X_val
df_test[new_cols] = X_test

final_feat_cols = new_cols
print(f"Final number of features F: {len(final_feat_cols)}")

class VentDataset(Dataset):
    def __init__(self, data_df, feat_cols, target_col):
        if not np.isin(np.unique(data_df["u_out"].values), [0, 1]).all():
            raise ValueError("u_out must be 0/1, otherwise the inhalation mask is invalid")
        self.x_list = []
        self.y_list = []
        self.mask_list = []
        for bid, group in data_df.sort_values(["breath_id", "time_step"]).groupby("breath_id"):
            x_seq = group[feat_cols].values.astype(np.float32)
            y_seq = group[target_col].values.astype(np.float32)
            mask_seq = group["u_out"].values.astype(np.float32)
            self.x_list.append(x_seq)
            self.y_list.append(y_seq)
            self.mask_list.append(mask_seq)
    def __len__(self):
        return len(self.x_list)
    def __getitem__(self, idx):
        x = torch.tensor(self.x_list[idx])
        y = torch.tensor(self.y_list[idx])
        mask = torch.tensor(self.mask_list[idx])
        return x, y, mask

train_ds = VentDataset(df_train, final_feat_cols, target_col)
val_ds = VentDataset(df_val, final_feat_cols, target_col)
test_ds = VentDataset(df_test, final_feat_cols, target_col)

BATCH_SIZE = 64
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=len(train_ds) >= BATCH_SIZE)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

x_sample, y_sample, mask_sample = next(iter(train_loader))
print(f"Input shape [B, T, F]: {x_sample.shape}")
print(f"Label shape [B, T]: {y_sample.shape}")
print(f"Mask shape [B, T]: {mask_sample.shape}")
print("First 10 mask values:", mask_sample[0,:10])



## (III) Build and train the LSTM baseline
Define a custom `MaskedMAELoss` that computes loss only over the inhalation interval (u_out=0). Build a single-layer unidirectional LSTM baseline with `batch_first=True`, predicting pressure at each time step.
Train for 30 epochs with Adam (gradients on during training, off during validation). Save the best model weights to a `.pt` file based on validation inhalation MAE.


In [ ]:
import torch.nn as nn
import time
import copy

class MaskedMAELoss(nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, pred, target, u_out_mask):
        valid_mask = (u_out_mask == 0)
        abs_err = torch.abs(pred - target)
        abs_err_valid = abs_err[valid_mask]
        if abs_err_valid.numel() == 0:
            return (pred * 0.0).sum()
        return torch.mean(abs_err_valid)

class LstmBaseline(nn.Module):
    def __init__(self, input_feat_dim, hidden_dim=64):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_feat_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            bidirectional=False,
            batch_first=True
        )
        self.linear = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        pred = self.linear(lstm_out)
        pred = pred.squeeze(-1)
        return pred

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_FEAT_DIM = len(final_feat_cols)
HIDDEN_DIM = 64
LR = 1e-3
EPOCHS = 30

model_lstm = LstmBaseline(input_feat_dim=INPUT_FEAT_DIM, hidden_dim=HIDDEN_DIM).to(device)
criterion = MaskedMAELoss()
optimizer = torch.optim.Adam(model_lstm.parameters(), lr=LR)

print(f"LSTM baseline parameters: {sum(p.numel() for p in model_lstm.parameters()):,}")
print(f"Model input feature dim INPUT_FEAT_DIM = {INPUT_FEAT_DIM}")

history_lstm = {
    "train_loss": [],
    "val_inhale_mae": [],
    "epoch_time": []
}
best_val_mae = float("inf")
best_model_weights = None



In [ ]:
best_val_mae = float('inf')
best_model_weights = None

for epoch in range(EPOCHS):
    t0 = time.time()
    model_lstm.train()
    total_train_loss = 0.0
    train_batch_count = 0
    for x, y, u_out in train_loader:
        x, y, u_out = x.to(device), y.to(device), u_out.to(device)
        optimizer.zero_grad()
        y_pred = model_lstm(x)
        loss = criterion(y_pred, y, u_out)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        train_batch_count += 1
    avg_train_loss = total_train_loss / train_batch_count

    model_lstm.eval()
    inhale_err_sum = 0.0
    inhale_cnt = 0
    with torch.no_grad():
        for x, y, u_out in val_loader:
            x, y, u_out = x.to(device), y.to(device), u_out.to(device)
            y_pred = model_lstm(x)
            valid_mask = (u_out == 0)
            inhale_err_sum += torch.sum(torch.abs(y_pred[valid_mask] - y[valid_mask])).item()
            inhale_cnt += torch.sum(valid_mask).item()
    if inhale_cnt > 0:
        val_inhale_mae = inhale_err_sum / inhale_cnt
    else:
        val_inhale_mae = float('inf')

    epoch_cost = time.time() - t0
    history_lstm["train_loss"].append(avg_train_loss)
    history_lstm["val_inhale_mae"].append(val_inhale_mae)
    history_lstm["epoch_time"].append(epoch_cost)

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | train_loss:{avg_train_loss:.4f} | val_inhale_MAE:{val_inhale_mae:.4f} | time:{epoch_cost:.2f}s")

    if val_inhale_mae < best_val_mae:
        best_val_mae = val_inhale_mae
        best_model_weights = {k: v.detach().cpu().clone() for k, v in model_lstm.state_dict().items()}

# Only load and save if optimal weights were found
if best_model_weights is not None:
    model_lstm.load_state_dict(best_model_weights)
    torch.save(best_model_weights, str(WEIGHT_DIR / "lstm_best.pt"))
    print(f"\nTraining finished, best validation inhalation MAE = {best_val_mae:.4f}")
else:
    print("\nWarning: no inhalation samples collected, model was not saved!")




## (IV) Build the CNN-LSTM hybrid model
Add Conv1d + ReLU before the LSTM to extract local temporal features: transpose B×T×F to B×F×T for convolution, then back to B×T×C for the LSTM.
Padding keeps the sequence length T unchanged, and pressure is predicted at each time step. The dataset, features, loss function and epoch count are kept identical to the LSTM baseline, and the best weights are saved.


In [ ]:
import torch.nn.functional as F

class CnnLstm(nn.Module):
    def __init__(self, input_feat_dim, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv1d(input_feat_dim, 32, kernel_size=3, stride=1, padding=1)
        self.lstm = nn.LSTM(32, hidden_dim, num_layers=1, bidirectional=False, batch_first=True)
        self.linear = nn.Linear(hidden_dim, 1)
    def forward(self, x):
        x = x.transpose(1, 2)
        x = F.relu(self.conv1(x))
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        pred = self.linear(out)
        pred = pred.squeeze(-1)
        return pred


model_cnnlstm = CnnLstm(input_feat_dim=INPUT_FEAT_DIM, hidden_dim=64).to(device)
optimizer = torch.optim.Adam(model_cnnlstm.parameters(), lr=LR)

print(f"CNN-LSTM parameters: {sum(p.numel() for p in model_cnnlstm.parameters()):,}")

history_cnnlstm = {
    "train_loss": [],
    "val_inhale_mae": [],
    "epoch_time": []
}
best_val_mae = float("inf")
best_model_weights = None



In [ ]:
for epoch in range(EPOCHS):
    t0 = time.time()

    model_cnnlstm.train()
    total_train_loss = 0.0
    train_batch_count = 0
    for x, y, u_out in train_loader:
        x, y, u_out = x.to(device), y.to(device), u_out.to(device)
        optimizer.zero_grad()
        y_pred = model_cnnlstm(x)
        loss = criterion(y_pred, y, u_out)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()
        train_batch_count += 1
    avg_train_loss = total_train_loss / train_batch_count

    model_cnnlstm.eval()
    inhale_err_sum = 0.0
    inhale_cnt = 0
    with torch.no_grad():
        for x, y, u_out in val_loader:
            x, y, u_out = x.to(device), y.to(device), u_out.to(device)
            y_pred = model_cnnlstm(x)
            valid_mask = (u_out == 0)
            inhale_err_sum += torch.sum(torch.abs(y_pred[valid_mask] - y[valid_mask])).item()
            inhale_cnt += torch.sum(valid_mask).item()
    if inhale_cnt > 0:
        val_inhale_mae = inhale_err_sum / inhale_cnt
    else:
        val_inhale_mae = float('inf')

    epoch_cost = time.time() - t0
    history_cnnlstm["train_loss"].append(avg_train_loss)
    history_cnnlstm["val_inhale_mae"].append(val_inhale_mae)
    history_cnnlstm["epoch_time"].append(epoch_cost)

    print(f"Epoch {epoch+1:2d}/{EPOCHS} | train_loss:{avg_train_loss:.4f} | val_inhale_MAE:{val_inhale_mae:.4f} | time:{epoch_cost:.2f}s")

    if val_inhale_mae < best_val_mae:
        best_val_mae = val_inhale_mae
        best_model_weights = {k: v.detach().cpu().clone() for k, v in model_cnnlstm.state_dict().items()}

if best_model_weights is not None:
    model_cnnlstm.load_state_dict(best_model_weights)
    torch.save(best_model_weights, str(WEIGHT_DIR / "cnn_lstm_best.pt"))
    print(f"\nTraining finished, best validation inhalation MAE = {best_val_mae:.4f}")
else:
    print("\nWarning: no inhalation samples collected, model was not saved!")




## (V) Model and hyperparameter comparison
Run controlled experiments: LSTM with hidden dim 32 vs 64; CNN-LSTM with kernel size 3 vs 5. All other hyperparameters and the dataset are kept identical.
`evaluate_all_metrics` computes validation inhalation MAE, full-sequence MAE and full-sequence RMSE; parameter counts and training time are measured, and a text table summarizing all validation results is printed to the console.


In [ ]:
import math

def evaluate_all_metrics(model, data_loader):
    model.eval()
    inhale_abs_sum = 0.0
    inhale_cnt = 0
    all_abs_sum = 0.0
    all_sq_sum = 0.0
    all_cnt = 0
    with torch.no_grad():
        for x, y, u_out in data_loader:
            x, y, u_out = x.to(device), y.to(device), u_out.to(device)
            pred = model(x)
            abs_err = torch.abs(pred - y)
            sq_err = torch.square(pred - y)

            mask_inhale = (u_out == 0)
            inhale_abs_sum += torch.sum(abs_err[mask_inhale]).item()
            inhale_cnt += torch.sum(mask_inhale).item()

            all_abs_sum += torch.sum(abs_err).item()
            all_sq_sum += torch.sum(sq_err).item()
            all_cnt += torch.numel(y)
    res = {}
    if inhale_cnt > 0:
        res["val_inhale_mae"] = inhale_abs_sum / inhale_cnt
    else:
        res["val_inhale_mae"] = float("inf")
    res["full_mae"] = all_abs_sum / all_cnt
    res["full_rmse"] = math.sqrt(all_sq_sum / all_cnt)
    return res

def count_params(model):
    return sum(p.numel() for p in model.parameters())



In [ ]:
# LSTM experiment over hidden dim: hidden_dim=32 / 64
lstm_exp_cfg = [
    {"name":"LSTM_h32", "hidden_dim":32},
    {"name":"LSTM_h64", "hidden_dim":64}
]

lstm_exp_result = []
for cfg in lstm_exp_cfg:
    print(f"\n===== Training {cfg['name']} =====")
    exp_model = LstmBaseline(input_feat_dim=INPUT_FEAT_DIM, hidden_dim=cfg["hidden_dim"]).to(device)
    opt = torch.optim.Adam(exp_model.parameters(), lr=LR)
    best_mae = float("inf")
    best_w = None
    t_start = time.time()
    for ep in range(EPOCHS):
        exp_model.train()
        for x,y,u in train_loader:
            x,y,u = x.to(device),y.to(device),u.to(device)
            opt.zero_grad()
            p = exp_model(x)
            loss = criterion(p,y,u)
            loss.backward()
            opt.step()
        exp_model.eval()
        met = evaluate_all_metrics(exp_model, val_loader)
        if met["val_inhale_mae"] < best_mae:
            best_mae = met["val_inhale_mae"]
            best_w = {k: v.detach().cpu().clone() for k, v in exp_model.state_dict().items()}
    total_time = time.time() - t_start
    if best_w is None:
        best_w = {k: v.detach().cpu().clone() for k, v in exp_model.state_dict().items()}
    exp_model.load_state_dict(best_w)
    metric = evaluate_all_metrics(exp_model, val_loader)
    lstm_exp_result.append({
        "model_name":cfg["name"],
        "val_inhale_mae":metric["val_inhale_mae"],
        "full_mae":metric["full_mae"],
        "full_rmse":metric["full_rmse"],
        "params":count_params(exp_model),
        "epoch_run":EPOCHS,
        "train_total_sec":round(total_time,2)
    })
print("LSTM hyperparameter experiments complete")
for item in lstm_exp_result:
    print(item)



In [ ]:
# CNN-LSTM experiment over kernel size: kernel_size=3 / 5, channels fixed at 32, LSTM hidden dim fixed at 64
class CnnLstmKernelExp(nn.Module):
    def __init__(self, input_feat_dim, kernel_size, hidden_dim=64, out_channel=32):
        super().__init__()
        self.conv1 = nn.Conv1d(input_feat_dim, out_channel, kernel_size=kernel_size, stride=1, padding=kernel_size//2)
        self.lstm = nn.LSTM(out_channel, hidden_dim, num_layers=1, bidirectional=False, batch_first=True)
        self.linear = nn.Linear(hidden_dim,1)
    def forward(self,x):
        x = x.transpose(1,2)
        x = F.relu(self.conv1(x))
        x = x.transpose(1,2)
        o,_ = self.lstm(x)
        pred = self.linear(o)
        return pred.squeeze(-1)

cnnlstm_exp_cfg = [
    {"name":"CNN-LSTM_k3","kernel":3},
    {"name":"CNN-LSTM_k5","kernel":5}
]
cnnlstm_exp_result = []
for cfg in cnnlstm_exp_cfg:
    print(f"\n===== Training {cfg['name']} =====")
    exp_model = CnnLstmKernelExp(input_feat_dim=INPUT_FEAT_DIM, kernel_size=cfg["kernel"], hidden_dim=64, out_channel=32).to(device)
    opt = torch.optim.Adam(exp_model.parameters(), lr=LR)
    best_mae = float("inf")
    best_w = None
    t_start = time.time()
    for ep in range(EPOCHS):
        exp_model.train()
        for x,y,u in train_loader:
            x,y,u = x.to(device),y.to(device),u.to(device)
            opt.zero_grad()
            p = exp_model(x)
            loss = criterion(p,y,u)
            loss.backward()
            opt.step()
        exp_model.eval()
        met = evaluate_all_metrics(exp_model, val_loader)
        if met["val_inhale_mae"] < best_mae:
            best_mae = met["val_inhale_mae"]
            best_w = {k: v.detach().cpu().clone() for k, v in exp_model.state_dict().items()}
    total_time = time.time() - t_start
    if best_w is None:
        best_w = {k: v.detach().cpu().clone() for k, v in exp_model.state_dict().items()}
    exp_model.load_state_dict(best_w)
    metric = evaluate_all_metrics(exp_model, val_loader)
    cnnlstm_exp_result.append({
        "model_name":cfg["name"],
        "val_inhale_mae":metric["val_inhale_mae"],
        "full_mae":metric["full_mae"],
        "full_rmse":metric["full_rmse"],
        "params":count_params(exp_model),
        "epoch_run":EPOCHS,
        "train_total_sec":round(total_time,2)
    })
print("CNN-LSTM hyperparameter experiments complete")
for item in cnnlstm_exp_result:
    print(item)



In [ ]:

print("="*110)
print(f"{'Model':<16}{'Val inhale MAE':<16}{'Full MAE':<12}{'Full RMSE':<12}{'Params':<10}{'Epochs':<8}{'Train sec'}")
print("-"*110)
for r in lstm_exp_result + cnnlstm_exp_result:
    print(f"{r['model_name']:<16}{r['val_inhale_mae']:<14.4f}{r['full_mae']:<12.4f}{r['full_rmse']:<12.4f}{r['params']:<12}{r['epoch_run']:<8}{r['train_total_sec']}")
print("="*110)



## (VI) Metrics and result analysis
Load the saved best weights of LSTM_h64 and CNN-LSTM_k3, evaluate on the **local test set that was not used for training**, and compute test inhalation MAE, full-sequence MAE and full-sequence RMSE.
Flatten the batch data, select 4 test breathing samples, and plot the true pressure, both models' predictions and absolute error curves, with green shading marking the inhalation interval.
Summarize the validation metrics of all 4 models into a markdown table for easy copying into a report.


In [ ]:
def test_metric_calc(model, test_loader):
    model.eval()
    inhale_abs_sum = 0.0
    inhale_cnt = 0
    all_abs_sum = 0.0
    all_sq_sum = 0.0
    all_cnt = 0
    pred_list = []
    label_list = []
    u_list = []
    with torch.no_grad():
        for x, y, u_out in test_loader:
            x, y, u_out = x.to(device), y.to(device), u_out.to(device)
            pred = model(x)
            pred_list.append(pred.cpu().numpy())
            label_list.append(y.cpu().numpy())
            u_list.append(u_out.cpu().numpy())
            abs_err = torch.abs(pred - y)
            sq_err = torch.square(pred - y)
            mask_inhale = (u_out == 0)
            inhale_abs_sum += torch.sum(abs_err[mask_inhale]).item()
            inhale_cnt += torch.sum(mask_inhale).item()
            all_abs_sum += torch.sum(abs_err).item()
            all_sq_sum += torch.sum(sq_err).item()
            all_cnt += torch.numel(y)
    res = {}
    res["test_inhale_mae"] = inhale_abs_sum / inhale_cnt if inhale_cnt>0 else float("inf")
    res["test_full_mae"] = all_abs_sum / all_cnt
    res["test_full_rmse"] = math.sqrt(all_sq_sum / all_cnt)
    return res, pred_list, label_list, u_list



In [ ]:
# Evaluate the best LSTM_h64 and CNN-LSTM_k3 on the test set
model_lstm_best = LstmBaseline(input_feat_dim=INPUT_FEAT_DIM, hidden_dim=64).to(device)
model_cnnlstm_best = CnnLstm(input_feat_dim=INPUT_FEAT_DIM, hidden_dim=64).to(device)
model_lstm_best.load_state_dict(torch.load(str(WEIGHT_DIR / "lstm_best.pt"), map_location=device, weights_only=True))
model_cnnlstm_best.load_state_dict(torch.load(str(WEIGHT_DIR / "cnn_lstm_best.pt"), map_location=device, weights_only=True))

lstm_test_metric, lstm_pred, lstm_y, lstm_u = test_metric_calc(model_lstm_best, test_loader)
cnnlstm_test_metric, cnn_pred, cnn_y, cnn_u = test_metric_calc(model_cnnlstm_best, test_loader)

print("===== Local test set metrics =====")
print(f"LSTM_h64 inhalation MAE:{lstm_test_metric['test_inhale_mae']:.4f}, full MAE:{lstm_test_metric['test_full_mae']:.4f}, full RMSE:{lstm_test_metric['test_full_rmse']:.4f}")
print(f"CNN-LSTM_k3 inhalation MAE:{cnnlstm_test_metric['test_inhale_mae']:.4f}, full MAE:{cnnlstm_test_metric['test_full_mae']:.4f}, full RMSE:{cnnlstm_test_metric['test_full_rmse']:.4f}")






In [ ]:
def flatten_batch_list(batch_list):
    res = []
    for batch in batch_list:
        for seq in batch:
            if isinstance(seq, np.ndarray):
                res.append(seq)
            else:
                res.append(seq.detach().cpu().numpy())
    return res

def plot_all_cycles(y_true_all, pred_lstm_all, pred_cnn_all, u_all):
    fig, axes = plt.subplots(4, 2, figsize=(14, 12))
    for i in range(4):
        yt = y_true_all[i].reshape(-1)
        pl = pred_lstm_all[i].reshape(-1)
        pc = pred_cnn_all[i].reshape(-1)
        u_flag = u_all[i].reshape(-1)

        x = np.arange(len(yt))
        ax0 = axes[i,0]
        ax0.plot(x, yt, 'k-', label="Ground truth")
        ax0.plot(x, pl, 'b-', label="LSTM_h64 pred")
        ax0.plot(x, pc, 'orange', label="CNN-LSTM_k3 pred")
        ax0.fill_between(x, 0, np.max(yt)+5, where=(u_flag==0), color="#7ccd7c", alpha=0.3, label="Inhalation")
        ax0.set_ylabel("Pressure")
        ax0.set_xlabel("Time step")
        ax0.legend(loc="upper right", fontsize=8)
        ax0.grid(alpha=0.3)

        ax1 = axes[i,1]
        err_lstm = np.abs(yt - pl)
        err_cnn = np.abs(yt - pc)
        ax1.plot(x, err_lstm, 'b-', label="LSTM error")
        ax1.plot(x, err_cnn, 'orange', label="CNN-LSTM error")
        ax1.set_ylabel("Absolute error")
        ax1.set_xlabel("Time step")
        ax1.legend(loc="upper right", fontsize=8)
        ax1.grid(alpha=0.3)
    plt.tight_layout()
    return fig

lstm_pred_flat = flatten_batch_list(lstm_pred)
lstm_y_flat = flatten_batch_list(lstm_y)
lstm_u_flat = flatten_batch_list(lstm_u)
cnn_pred_flat = flatten_batch_list(cnn_pred)

fig = plot_all_cycles(lstm_y_flat, lstm_pred_flat, cnn_pred_flat, lstm_u_flat)
fig.savefig(str(FIG_DIR / "four_samples.png"), dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

all_result = lstm_exp_result + cnnlstm_exp_result
result_df = pd.DataFrame(all_result)
try:
    print(result_df.to_markdown(index=False, floatfmt=".4f"))
except ImportError:
    print(result_df.to_string(index=False))



